# Preprocess fetal dataset for trajectory analysis

**Developed by:** Anna Maguza\
**Affiliation:** Faculty of Medicine, Würzburg University\
**Creation date:** 6th March 2025\
**Last modified date:** 6th March 2025

In this notebook, we:
+ The cell index is not unique for some cells, we need to fix this issue
+ Transfer spliced/unspliced counts to fetal dataset 

## Import packages

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import json
from datetime import datetime

In [2]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

## Load dataset

In [3]:
adata_full_annotated = sc.read_h5ad("data/gut_data/gut_hs_all_datasets_full_annotated_AM_03032025_141544_raw.h5ad")

In [ ]:
adata_velocity = sc.read_h5ad("data/gut_data/gut_hs_merged_datasets_AM_18122024_131905_raw.h5ad")

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [5]:
fetalSC = sc.read_h5ad("data/gut_data/gut_hs_fetalSC_AM_05032025_150941_raw.h5ad")

## Fix cell indexes

+ From annotated object

In [6]:
adata_full_annotated.obs['cell_index'].duplicated().sum()

197060

In [7]:
adata_full_annotated.obs['cell_index'] = adata_full_annotated.obs['cell_index'].astype(str) + '_'+ adata_full_annotated.obs['ENA_RUN'].astype(str)

In [8]:
adata_full_annotated.obs['cell_index'].duplicated().sum()

21534

In [9]:
adata_full_annotated

AnnData object with n_obs × n_vars = 387076 × 43704
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparation_pro

In [10]:
adata_full_annotated = adata_full_annotated[~adata_full_annotated.obs['cell_index'].duplicated()].copy()

In [11]:
adata_full_annotated.obs['cell_index'].duplicated().sum()

0

In [12]:
adata_full_annotated

AnnData object with n_obs × n_vars = 365542 × 43704
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparation_pro

* From old velocity object

In [13]:
adata_velocity.obs['predicted_doublets'].value_counts()

predicted_doublets
False    375413
None      13934
True      12996
Name: count, dtype: int64

In [14]:
adata_velocity = adata_velocity[adata_velocity.obs['predicted_doublets'] != 'True']

In [15]:
adata_velocity.obs['cell_index'] = adata_velocity.obs['cell_id'].copy()

/var/folders/kr/nd4y_1_s34n42lrht8wdh4cr0000gq/T/ipykernel_30803/2977322411.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_velocity.obs['cell_index'] = adata_velocity.obs['cell_id'].copy()
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [16]:
adata_velocity.obs['cell_index'].duplicated().sum()

198232

In [17]:
adata_velocity.obs['cell_index'] = adata_velocity.obs['cell_index'].astype(str) + '_'+ adata_velocity.obs['ENA_RUN'].astype(str)

In [18]:
adata_velocity.obs['cell_index'].duplicated().sum()

21700

In [19]:
adata_velocity = adata_velocity[~adata_velocity.obs['cell_index'].duplicated()].copy()

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [20]:
adata_velocity.obs['cell_index'].duplicated().sum()

0

* From fetal SC

In [21]:
fetalSC.obs['cell_index'].duplicated().sum()

3908

In [22]:
fetalSC.obs['cell_index'] = fetalSC.obs['cell_index'].astype(str) + '_'+ fetalSC.obs['ENA_RUN'].astype(str)

In [23]:
fetalSC.obs['cell_index'].duplicated().sum()

0

In [24]:
fetalSC = fetalSC[~fetalSC.obs['cell_index'].duplicated()].copy()

In [25]:
fetalSC.obs['cell_index'].duplicated().sum()

0

### Transfer cell states to the big object

In [26]:
adata_velocity.obs.index = adata_velocity.obs['cell_index']
adata_full_annotated.obs.index = adata_full_annotated.obs['cell_index']
fetalSC.obs.index = fetalSC.obs['cell_index']

In [27]:
adata_full_annotated.obs.index.isin(adata_velocity.obs.index).sum()

365542

In [28]:
adata_velocity = adata_velocity[adata_velocity.obs_names.isin(adata_full_annotated.obs_names)]

In [29]:
columns_to_transfer = [
    'celltype', 
    'library_construnction_and_layout', 
    'cell_states', 
    'cellstates_scANVI', 
    'confidence_score', 
    'gut_region'
]

for col in columns_to_transfer:
    adata_velocity.obs[col] = adata_full_annotated[adata_velocity.obs_names].obs[col].values

/var/folders/kr/nd4y_1_s34n42lrht8wdh4cr0000gq/T/ipykernel_30803/678649446.py:11: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_velocity.obs[col] = adata_full_annotated[adata_velocity.obs_names].obs[col].values


In [30]:
columns_to_transfer = [
    'leiden_cluster'
]

In [31]:
for col in columns_to_transfer:
    if col in fetalSC.obs.columns:
        new_column = pd.Series(index=adata_velocity.obs.index, dtype='object')
        common_indices = set(fetalSC.obs.index).intersection(set(adata_velocity.obs.index))
        for idx in common_indices:
            new_column[idx] = fetalSC.obs.loc[idx, col]
        adata_velocity.obs[col] = new_column

## Save datasets

In [23]:
current_history = adata_full_annotated.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'Fixed doublicated cell indices',
})
current_history.append(new_entry)

adata_full_annotated.uns['processing_history'] = current_history

In [24]:
project = 'gut'
species = 'hs'
name = 'AM'
counts = 'raw'
atribute = 'full_annotated'

adata_full_annotated.write_h5ad(f"data/gut_data/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}.h5ad")

In [36]:
current_history = adata_velocity.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'Fixed doublicated cell indices, merge annotations',
})
current_history.append(new_entry)

adata_velocity.uns['processing_history'] = current_history

In [37]:
project = 'gut'
species = 'hs'
name = 'AM'
counts = 'raw_valocity'
atribute = 'all_full_annotated'

adata_velocity.write_h5ad(f"data/gut_data/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}.h5ad")

## Extract fetal

In [40]:
adata_fetal = adata_velocity[adata_velocity.obs['age_group'].isin(['second trimester', 'first trimester'])].copy()

In [44]:
adata_fetal_epi = adata_fetal[adata_fetal.obs['celltype'].isin(['Epithelial'])].copy()

In [43]:
project = 'gut'
species = 'hs'
name = 'AM'
counts = 'raw_velocity'
atribute = 'fetal'

adata_fetal.write_h5ad(f"data/gut_data/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}.h5ad")

In [46]:
project = 'gut'
species = 'hs'
name = 'AM'
counts = 'raw_velocity'
atribute = 'fetal_epithelial'

adata_fetal_epi.write_h5ad(f"data/gut_data/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}.h5ad")